## 1. Install Dependencies

In [ ]:
# tested on python 3.12.10
import zipfile

!curl -L https://github.com/mpj1234/ncnn-yolo26-android/releases/download/asserts/ultralytics-8.4.6.zip -o ultralytics.zip
with zipfile.ZipFile("ultralytics.zip", "r") as z:
    z.extractall("ultralytics-src")
%pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu128
%pip install -U ncnn==1.0.20260114 pnnx==20260112 roboflow
%pip install -e ./ultralytics-src

In [ ]:
# tested on python 3.12.10
%pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu128
%pip install -U ultralytics roboflow ncnn==1.0.20260114 pnnx==20260112


## 2. Download and prepare dataset

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="Mt0m4dCKTTlNf29OV3zm")
project = rf.workspace("dylans-workspace-3init").project("ping-pong-detection-0guzq-f7zly")
version = project.version(1)
dataset = version.download("yolo26") # This downloads the images AND the data.yaml
                

### 2.1 Patch dataset with missing labels

In [ ]:
from pathlib import Path
import re
import yaml

COCO_NAMES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
    "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear",
    "hair drier", "toothbrush",
]


dataset_root = Path("Ping-Pong-Detection-1")
def rewrite_data_yaml(dry_run: bool = False):
    path = dataset_root / "data.yaml"
    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    original_names = data.get("names", [])
    print(f"Original names: {original_names}")

    # Force dataset to YOLO COCO label space so class id 32 is 'sports ball'.
    data["names"] = COCO_NAMES
    data["nc"] = len(COCO_NAMES)

    if dry_run:
        print("Dry run: data.yaml would be rewritten with COCO names.")
        return

    path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print(f"Updated: {path}")

def rewrite_label_file(path: Path, target_class: int, dry_run: bool = False):
    changed_lines = 0
    original = path.read_text(encoding="utf-8").splitlines(keepends=True)
    updated = []

    for line in original:
        stripped = line.strip()

        if not stripped:
            updated.append(line)
            continue

        parts = stripped.split(maxsplit=1)
        first = parts[0]
        rest = parts[1] if len(parts) > 1 else ""

        # Accept integer-like or float-like ids and normalize to class 32.
        if not re.fullmatch(r"[-+]?\d+(?:\.\d+)?", first):
            updated.append(line)
            continue

        new_line = f"{target_class} {rest}".rstrip()
        if line.endswith("\n"):
            new_line += "\n"

        if new_line != line:
            changed_lines += 1
        updated.append(new_line)

    if changed_lines and not dry_run:
        path.write_text("".join(updated), encoding="utf-8")

    return changed_lines

def change_label_classes(dry_run: bool = True):
    target_class = 32
    label_dirs = [
        dataset_root / "train" / "labels",
        dataset_root / "valid" / "labels",
        dataset_root / "test" / "labels",
    ]

    total_files = 0
    touched_files = 0
    total_lines_changed = 0

    for label_dir in label_dirs:
        if not label_dir.exists():
            print(f"Skipping missing directory: {label_dir}")
            continue

        for txt_file in label_dir.rglob("*.txt"):
            total_files += 1
            changed = rewrite_label_file(txt_file, target_class, dry_run=dry_run)
            if changed:
                touched_files += 1
                total_lines_changed += changed
                print(f"Updated {txt_file} ({changed} line(s))")

    mode = "DRY RUN" if dry_run else "WRITE"
    print(f"\n[{mode}] Scanned files: {total_files}")
    print(f"[{mode}] Files changed: {touched_files}")
    print(f"[{mode}] Label lines changed: {total_lines_changed}")


dry_run = False  # Set to True to preview only
rewrite_data_yaml(dry_run=dry_run)
change_label_classes(dry_run=dry_run)

# 3. Train

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from ultralytics import YOLO, settings
settings.reset() # reset the output dirs if different envs messed with it


# 1. Load a pretrained YOLO11n model
model = YOLO('yolo26n.pt')

# 2. Train the model
# 'data.yaml' contains paths to your orange ball images and class names
model.train(data="Ping-Pong-Detection-1/data.yaml", epochs=100, imgsz=640,batch=32)

## 3. Export YOLO26 NCNN

In [ ]:
train_dir = "runs/detect/train7/weights"

In [ ]:
from pathlib import Path
from ultralytics import YOLO
YOLO(Path(train_dir)/"best.pt").export(**{
    'format': 'ncnn',
    'opset': 20,
    'simplify': True,
    'batch': 1,
    'imgsz': 640,
})

### 3.1 Copy the exported model to app assets

In [ ]:
import shutil
from pathlib import Path

assets_dir = Path("app") / "src" / "main" / "assets"
renames = {
    "model.ncnn.bin": "yolo26n.ncnn.bin",
    "model.ncnn.param": "yolo26n.ncnn.param",
}
for src, dst in renames.items():
    shutil.copy((Path(train_dir)/"best_ncnn_model"/src).resolve(), (assets_dir / dst).resolve())
    print(f"Copied {src} -> {assets_dir / dst}")